In [1]:
import torch
import torch.nn as nn
from torchinfo import summary
from sklearn.model_selection import train_test_split

In [2]:
X = torch.rand(10,5)
y = torch.tensor([1,0,1,0,1,0,1,0,1,0], dtype=torch.float32)

In [ ]:
# sequential way of building a model
# we can also build a model using nn.Sequential, which is a container module that sequences together a series of modules.
# the nn.Sequential class allows us to build a model by simply stacking layers together in a sequential manner.
class nn_model(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.model = nn.Sequential(nn.Linear(num_features,3),
                                 nn.ReLU(),
                                 nn.Linear(3,1),
                                 nn.Sigmoid())

  def forward(self, x):
    x = self.model(x)
    return x

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

model = nn_model(num_features=X.shape[1])
loss_fn = nn.BCELoss()
summary(model, input_size=(1,X.shape[1]))

Layer (type:depth-idx)                   Output Shape              Param #
nn_model                                 [1, 1]                    --
├─Sequential: 1-1                        [1, 1]                    --
│    └─Linear: 2-1                       [1, 3]                    18
│    └─ReLU: 2-2                         [1, 3]                    --
│    └─Linear: 2-3                       [1, 1]                    4
│    └─Sigmoid: 2-4                      [1, 1]                    --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

- 5 -> 3 -> 1
- 5*3 + 3*1 = 15 + 3 = 18
- biases = 3 + 1 = 4
- total parameters = 18 + 4 = 22


In [11]:
learning_rate = 0.01
epochs = 10

In [12]:
for epoch in range(epochs):

    # forward pass
    y_pred = model(X_train)
    # calculate the loss
    loss = loss_fn(y_pred, y_train.view(-1,1))
    # backward pass
    loss.backward()
    # update the weights
    with torch.no_grad():
      for param in model.parameters():
        param -= learning_rate * param.grad
    # zero the gradients
    model.zero_grad()   

    print(f'Epoch : {epoch+1}/{epochs} | Loss : {loss.item():.4f}')

Epoch : 1/10 | Loss : 0.7250
Epoch : 2/10 | Loss : 0.7248
Epoch : 3/10 | Loss : 0.7245
Epoch : 4/10 | Loss : 0.7243
Epoch : 5/10 | Loss : 0.7241
Epoch : 6/10 | Loss : 0.7238
Epoch : 7/10 | Loss : 0.7236
Epoch : 8/10 | Loss : 0.7234
Epoch : 9/10 | Loss : 0.7231
Epoch : 10/10 | Loss : 0.7229


In [19]:
model.state_dict()

OrderedDict([('model.0.weight',
              tensor([[-0.3152, -0.0445,  0.3606, -0.2569, -0.2610],
                      [-0.0221,  0.2446, -0.2973,  0.0913, -0.0705],
                      [ 0.2991,  0.3399, -0.0481,  0.0437, -0.2267]])),
             ('model.0.bias', tensor([ 0.1307, -0.0827, -0.2656])),
             ('model.2.weight', tensor([[-0.3198,  0.5684, -0.1338]])),
             ('model.2.bias', tensor([-0.5135]))])

In [22]:
with torch.no_grad():
    # no_grad() is a context manager that disables gradient calculation, which is useful for inference and evaluation.
    # it reduces memory consumption and speeds up computations since gradients are not needed during inference.
    # we use no_grad() when we are not training the model, but just want to make predictions or evaluate the model on a test set.
  y_pred = model(X_test)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test.view(-1,1)).float().mean()
  print(f'Accuracy : {accuracy}')

Accuracy : 0.5
